# Hardware timing comparison � eight trials per rank

This self-contained notebook acquires **rank 1 first, then rank 58**, with
**eight separate hardware acquisitions per rank**. Each acquisition is one
IBM Sampler job containing three circuits, so the default run submits
**16 jobs / 48 circuit executions**. Each method therefore has eight timing
records per rank; `SHOTS` is a separate setting.

The comparisons are

1. **Sequential classical post-processing versus sequential feed-forward.**
2. **Additional parallelisation**, comparing sequential and parallelised classical post-processing.

Defaults are `ibm_pittsburgh`, R3, optimisation level 2 and no DD. The resource,
decoders and channel assignments follow the four supplied notebooks; the
R4/R5 classical builder is adapted to the same R3 / 19-qubit problem.
This is a new matched benchmark, not a reproduction of the old acquisitions.

**Nothing is submitted by default.** Configure the next cell and change
`RUN_HARDWARE = True` only when you intend to submit the 16 jobs. The notebook
uses an existing saved IBM account. Do not paste an API token into it.

Timings come from [IBM's hardware-compiler scheduling traces](https://quantum.cloud.ibm.com/docs/en/guides/qiskit-runtime-circuit-timing).
The reported duration is `max(T0 + Duration) * backend.dt`, including reported
initialisation/finalisation, readout and classical-control communication.
It is **not** queue time, billed QPU time, offline CPU time or a per-shot stopwatch
measurement. It also differs in scope from the old synthetic preparation-prefix
values, which are not reused. Frozen circuits can return identical timings
across the eight jobs; averaging them does not measure per-shot latency variance.


In [ ]:
from __future__ import annotations

import argparse

import csv

import hashlib

import importlib.metadata

import io

import json

import math

import statistics

import sys

from collections import Counter

from datetime import datetime, timezone

from pathlib import Path

VARIANTS = ("sequential_feed_forward", "sequential_classical", "parallel_classical")

ROLES = {1: ((2, 0), (3, 1)), 58: ((0, 4), (1, 3))}

ACTIVE = {1: (0, 1, 4), 58: (2, 3, 4)}

TIMING_GUIDE = "https://quantum.cloud.ibm.com/docs/en/guides/qiskit-runtime-circuit-timing"

METRIC = "hardware_compiler_total_schedule_us"

SOURCE_NAMES = (
    "original_MCM_SHARED_PREPARATION_RANK1_TEST_NEWv7_V5_M3_DIRECT_CHSH (1).ipynb",
    "original_MCM_SHARED_PREPARATION_RANK58_TEST_NEWv7_V5_M3_DIRECT_CHSH (1).ipynb",
    "NEWv4_MCM_CLASSICAL_R456_RANK1.ipynb",
    "NEWv4_MCM_CLASSICAL_R456_RANK58.ipynb",
)

from IPython.display import Markdown, display


## 1. Configuration

`N_TRIALS = 8` means eight hardware jobs **per rank**, with all three methods
in each job. Odd/even trials alternate the requested method order. The circuits
and compilation target are frozen across repetitions; both ranks share the
same initial physical assignment. The sequential classical circuit retains
the routed feed-forward backbone with conditional correction blocks removed.

No M3 calibration or full tomography jobs are needed for this timing-only
benchmark. The selected basis supplies a single reference�receiver correlation
as a sanity check, **not** an entanglement-fidelity estimate. For ordinary use,
leave the default settings except your account/instance if required.


In [ ]:
BACKEND_NAME = "ibm_pittsburgh"
N_TRIALS = 8
ROUNDS = 3
OPTIMIZATION_LEVEL = 2
SHOTS = 256                  # shots per circuit; independent of N_TRIALS
ANALYSIS_BASIS = "ZZ"         # one basis only, not full tomography
SEED_TRANSPILER = 42

ACCOUNT_NAME = None           # optional existing saved account name
RUNTIME_INSTANCE = None       # optional instance CRN, not an API token
INITIAL_LAYOUT = None         # optional list of 19 distinct physical indices
REP_DELAY_SECONDS = None      # optional common inter-shot delay
OUTPUT_DIRECTORY = None       # None => ./timing_runs/<UTC timestamp>

RUN_OFFLINE_TESTS = False     # True runs ideal simulations; no IBM connection
RUN_HARDWARE = False          # change deliberately to True to submit 16 jobs

RUN_RECOVERY = False          # re-fetch existing job IDs only; never resubmit
RECOVER_RUN_DIRECTORY = None  # path to an existing run containing manifest.json

# Preserve the most recent result path when rerunning display cells.
LAST_RUN_DIRECTORY = globals().get("LAST_RUN_DIRECTORY", None)


## 2. Implementation

The following cells contain all required code, so the notebook can be copied
and run without the `.py` file or the original notebooks. Implementation cells
may be collapsed by your Jupyter interface. They only define functions.


In [ ]:
def utc_now():
    return datetime.now(timezone.utc).isoformat()


def source_directory():
    """Support both a standalone script and embedded Jupyter definitions."""
    source = globals().get("__file__")
    return Path(source).resolve().parent if source else Path.cwd().resolve()


def write_json(path, obj):
    """Write only inside the explicitly created run directory."""
    from qiskit_ibm_runtime import RuntimeEncoder

    path.write_text(json.dumps(obj, cls=RuntimeEncoder, indent=2), encoding="utf-8")


def write_csv(path, records):
    if not records:
        return
    fields = list(dict.fromkeys(key for row in records for key in row))
    with path.open("w", newline="", encoding="utf-8") as stream:
        writer = csv.DictWriter(stream, fieldnames=fields)
        writer.writeheader()
        writer.writerows(records)


def dependencies():
    """No account access. Keep --help available without quantum packages."""
    from packaging.version import Version

    versions = {name: importlib.metadata.version(name)
                for name in ("qiskit", "qiskit-ibm-runtime")}
    if Version(versions["qiskit"]) < Version("2.1"):
        raise RuntimeError("Use Qiskit >= 2.1 for this script.")
    if Version(versions["qiskit-ibm-runtime"]) < Version("0.43.0"):
        raise RuntimeError("IBM scheduler timing requires qiskit-ibm-runtime >= 0.43.0.")
    return versions


### AME resource, decoder circuits and the three implementations


In [ ]:
def ame_prepare(circuit, q):
    circuit.h(q[2])
    circuit.h(q[3])
    circuit.cx(q[2], q[1])
    circuit.cx(q[1], q[0])
    circuit.h(q[2])
    circuit.cx(q[2], q[4])
    circuit.cx(q[3], q[2])
    circuit.cx(q[2], q[0])


def decode(circuit, resource, rank):
    q = [resource[index] for index in ACTIVE[rank]]
    if rank == 1:
        circuit.h(q[0])
        circuit.s(q[0])
        circuit.h(q[1])
        circuit.cx(q[0], q[1])
        circuit.sdg(q[0])
        circuit.h(q[0])
        circuit.cx(q[1], q[2])
        circuit.cx(q[0], q[1])
    else:
        circuit.h(q[0])
        circuit.x(q[1])
        circuit.x(q[2])
        circuit.cx(q[1], q[2])
        circuit.sdg(q[2])
        circuit.h(q[2])
        circuit.h(q[1])
        circuit.s(q[1])
        circuit.cx(q[0], q[2])
        circuit.sdg(q[2])
        circuit.h(q[2])
        circuit.h(q[0])
        circuit.s(q[0])
        circuit.cx(q[1], q[2])
        circuit.sdg(q[2])
        circuit.h(q[2])


def build_circuit(rank, variant, rounds=3, basis="ZZ"):
    """Logical circuit using standard measure for SDK routing.

    All readout instructions are converted to native measure_2 AFTER
    routing and optimization, BEFORE IBM scheduling/submission.
    """
    from qiskit import QuantumCircuit, QuantumRegister, ClassicalRegister

    if rank not in ROLES or variant not in VARIANTS:
        raise ValueError("Unknown rank or variant.")
    refs = QuantumRegister(2, "ref")
    inputs = QuantumRegister(2, "input")
    resources = [QuantumRegister(5, f"r{r}") for r in range(1, rounds + 1)]
    syndromes = [[ClassicalRegister(2, f"r{r}_m{ch}") for ch in "AB"]
                 for r in range(1, rounds + 1)]
    readout = ClassicalRegister(4, "analysis")
    circuit = QuantumCircuit(refs, inputs, *resources,
                             *(bits for pair in syndromes for bits in pair), readout,
                             name=f"rank{rank}_{variant}_R{rounds}")
    for ref, inp in zip(refs, inputs):
        circuit.h(ref)
        circuit.cx(ref, inp)
    current = list(inputs)
    bell = []
    if variant == "parallel_classical":
        for resource in resources:
            ame_prepare(circuit, resource)
            decode(circuit, resource, rank)
        for resource, pair in zip(resources, syndromes):
            for channel, (bits, (source, receiver)) in enumerate(zip(pair, ROLES[rank])):
                bell.append((current[channel], resource[source], bits[0], bits[1]))
                current[channel] = resource[receiver]
        assert len({q for inp, source, _, _ in bell for q in (inp, source)}) == 4 * rounds
        for inp, source, _, _ in bell:
            circuit.cx(inp, source)
        for inp, _, _, _ in bell:
            circuit.h(inp)
        circuit.barrier(label="TIMING_PARALLEL_READOUT_LAYER")
        for inp, source, u, v in bell:
            circuit.measure(inp, u)
            circuit.measure(source, v)
    else:
        circuit.barrier(label="TIMING_ROUND_0")
        for r, (resource, pair) in enumerate(zip(resources, syndromes), 1):
            ame_prepare(circuit, resource)
            decode(circuit, resource, rank)
            for channel, (bits, (source, receiver)) in enumerate(zip(pair, ROLES[rank])):
                inp, sender, output = current[channel], resource[source], resource[receiver]
                circuit.cx(inp, sender)
                circuit.h(inp)
                circuit.measure(inp, bits[0])
                circuit.measure(sender, bits[1])
                if variant == "sequential_feed_forward":
                    with circuit.if_test((bits[1], 1)):
                        circuit.x(output)
                    with circuit.if_test((bits[0], 1)):
                        circuit.z(output)
                current[channel] = output
            circuit.barrier(label=f"TIMING_ROUND_{r}")
    circuit.barrier(label="TIMING_ANALYSIS_START")
    for channel in range(2):
        for q in (refs[channel], current[channel]):
            if basis == "XX":
                circuit.h(q)
            elif basis == "YY":
                circuit.sdg(q)
                circuit.h(q)
            elif basis != "ZZ":
                raise ValueError("Use ZZ, XX, or YY analysis.")
        circuit.measure(refs[channel], readout[2 * channel])
        circuit.measure(current[channel], readout[2 * channel + 1])
    circuit.metadata = dict(rank=rank, variant=variant, rounds=rounds, basis=basis,
                            no_dd=True, logical_qubits=4 + 5 * rounds,
                            analysis_indices=[circuit.find_bit(bit).index for bit in readout],
                            syndrome_indices=[[[circuit.find_bit(bit).index for bit in bits]
                                               for bits in pair] for pair in syndromes])
    return circuit


### Matched routing, native readout and scheduling


In [ ]:
def remove_conditionals(circuit):
    """Preserve the routed native quantum backbone and final readout mapping."""
    out = circuit.copy()
    kept = []
    for item in circuit.data:
        if item.operation.name == "if_else":
            blocks = item.operation.blocks
            if len(blocks) != 1 or any(inst.operation.num_qubits > 1
                                       or inst.operation.num_clbits for inst in blocks[0].data):
                raise RuntimeError("Refusing to remove an unexpected conditional block.")
        else:
            if getattr(item.operation, "blocks", ()):
                raise RuntimeError("Unexpected control flow.")
            kept.append(item)
    out.data = kept
    out.name = circuit.name.replace("sequential_feed_forward", "sequential_classical")
    out.metadata = dict(circuit.metadata or {}, variant="sequential_classical",
                        routed_backbone_shared_with_feed_forward=True)
    return out


def expand_barriers(circuit):
    """Round boundaries must also stop SWAPs using routing ancillas."""
    from qiskit.circuit import Barrier, CircuitInstruction

    out = circuit.copy()
    used = set(q for item in out.data if item.operation.name != "barrier" for q in item.qubits)
    ordered = tuple(q for q in out.qubits if q in used)
    out.data = [CircuitInstruction(Barrier(len(ordered), label=item.operation.label), ordered, ())
                if item.operation.name == "barrier" else item for item in out.data]
    return out


def native_readout(circuit):
    from qiskit_ibm_runtime.circuit import MidCircuitMeasure

    out = circuit.copy()
    out.data = [item.replace(operation=MidCircuitMeasure())
                if item.operation.name == "measure" else item for item in out.data]
    return out


def circuit_stats(circuit):
    counts = Counter()
    conditional = 0
    for item in circuit.data:
        counts[item.operation.name] += 1
        for block in getattr(item.operation, "blocks", ()):
            conditional += len(block.data)
    used = sorted({circuit.find_bit(q).index for item in circuit.data
                   if item.operation.name not in ("delay", "barrier") for q in item.qubits})
    return dict(num_qubits=circuit.num_qubits, num_clbits=circuit.num_clbits,
                native_operations=dict(counts), conditional_body_instructions=conditional,
                used_physical_qubits=used)


def compile_rank(backend, rank, args, common_layout):
    from qiskit import transpile
    from qiskit.transpiler import PassManager
    from qiskit_ibm_runtime.transpiler.passes.scheduling import ALAPScheduleAnalysis, PadDelay

    if "measure_2" not in backend.target.operation_names:
        raise RuntimeError("Backend has no native measure_2. No silent readout substitution allowed.")
    logical_ff = build_circuit(rank, VARIANTS[0], args.rounds, args.analysis_basis)
    route_args = dict(backend=backend, optimization_level=0, routing_method="sabre",
                      seed_transpiler=args.seed, approximation_degree=1.0)
    if common_layout is not None:
        route_args["initial_layout"] = common_layout
    else:
        route_args["layout_method"] = "sabre"
    routed_ff = transpile(logical_ff, **route_args)
    initial_layout = list(routed_ff.layout.initial_index_layout(filter_ancillas=True))

    def optimize(routed):
        routed = expand_barriers(routed)
        optimized = transpile(routed, backend=backend, optimization_level=args.optimization_level,
                              initial_layout=list(range(routed.num_qubits)), routing_method="none",
                              seed_transpiler=args.seed, approximation_degree=1.0)
        return native_readout(expand_barriers(optimized))

    native_ff = optimize(routed_ff)
    # Remove blocks AFTER routing and optimization, so sequential schedules
    # differ only by corrections and subsequent padding/scheduling.
    native_classical = remove_conditionals(native_ff)
    logical_parallel = build_circuit(rank, VARIANTS[2], args.rounds, args.analysis_basis)
    parallel_route_args = dict(route_args, initial_layout=initial_layout)
    parallel_route_args.pop("layout_method", None)
    native_parallel = optimize(transpile(logical_parallel, **parallel_route_args))
    native = dict(zip(VARIANTS, (native_ff, native_classical, native_parallel)))
    assert native_ff.count_ops().get("if_else", 0) == 4 * args.rounds
    compiled = {}
    for variant, circuit in native.items():
        assert circuit.count_ops().get("measure_2", 0) == 4 * args.rounds + 4
        pm = PassManager([ALAPScheduleAnalysis(target=backend.target),
                          PadDelay(target=backend.target, fill_very_end=False,
                                   schedule_idle_qubits=False)])
        compiled[variant] = pm.run(circuit)
        compiled[variant].metadata = dict(circuit.metadata or {}, rank=rank, variant=variant,
                                          initial_layout=initial_layout,
                                          optimization_level=args.optimization_level,
                                          seed_transpiler=args.seed)
    return initial_layout, compiled


### Timing extraction, eight-trial summaries and Pauli-frame analysis


In [ ]:
def parse_timing(raw, dt_seconds):
    """IBM six-field format; T0 is absolute within the complete schedule.

    Never sum gate durations, add mutually exclusive branches, discard
    Hub/Receive channels, or treat every branch as starting at t=0.
    Both the documented list-of-lines and demonstrated newline-string
    representations are accepted. Units are verified against backend.dt.
    """
    if not math.isfinite(dt_seconds) or dt_seconds <= 0:
        raise ValueError("A positive hardware backend.dt is required.")
    if isinstance(raw, str):
        lines = raw.splitlines()
    elif isinstance(raw, (list, tuple)) and all(isinstance(line, str) for line in raw):
        lines = [line for group in raw for line in group.splitlines()]
    else:
        raise ValueError("Unknown IBM timing representation; raw metadata has been saved.")
    rows = []
    for line in lines:
        if not line.strip():
            continue
        words = next(csv.reader([line], skipinitialspace=True))
        if len(words) != 6:
            raise ValueError(f"Expected 6 timing fields, got {len(words)}: {line!r}")
        branch, instruction, channel, t0, duration, pulse = (s.strip() for s in words)
        start, length = int(t0), int(duration)
        if start < 0 or length < 0:
            raise ValueError("Negative hardware timing is unsupported.")
        rows.append(dict(branch=branch, instruction=instruction, channel=channel,
                         start_dt=start, duration_dt=length, end_dt=start + length,
                         pulse=pulse))
    if not rows or not any(row["duration_dt"] > 0 for row in rows):
        raise ValueError("Empty or zero-duration hardware timing trace.")
    end = max(row["end_dt"] for row in rows)
    start = min(row["start_dt"] for row in rows)
    return dict(duration_dt=end, duration_us=end * dt_seconds * 1e6,
                first_reported_start_dt=start,
                reported_envelope_us=(end - start) * dt_seconds * 1e6,
                instruction_rows=rows,
                branch_names=sorted({row["branch"] for row in rows}),
                communication_rows=sum(row["channel"] in ("Hub", "Receive") for row in rows))


def summarize(run_dir, manifest):
    records = manifest.get("records", [])
    write_csv(run_dir / "timings_per_trial.csv", records)
    means, effects = [], []
    # Older two-trial manifests remain recoverable with their original count.
    n_trials = int(manifest["settings"].get("trials", 2))
    expected = set(range(1, n_trials + 1))
    for rank in (1, 58):
        grouped = {}
        for variant in VARIANTS:
            chosen = [r for r in records if r["rank"] == rank and r["variant"] == variant]
            if {r["trial"] for r in chosen} != expected or len(chosen) != n_trials:
                continue
            values = [r["duration_us"] for r in chosen]
            grouped[variant] = {r["trial"]: r["duration_us"] for r in chosen}
            means.append(dict(rank=rank, variant=variant, trials=n_trials, metric=METRIC,
                              mean_us=statistics.mean(values), min_us=min(values), max_us=max(values)))
        if len(grouped) == 3:
            for effect, before, after in (
                ("classical_post_processing_vs_feed_forward", VARIANTS[0], VARIANTS[1]),
                ("additional_parallelisation", VARIANTS[1], VARIANTS[2]),
            ):
                b = statistics.mean(grouped[before].values())
                a = statistics.mean(grouped[after].values())
                effects.append(dict(rank=rank, effect=effect, before=before, after=after,
                                    before_mean_us=b, after_mean_us=a,
                                    mean_saved_us=statistics.mean(grouped[before][i] - grouped[after][i]
                                                                 for i in expected),
                                    reduction_of_mean_percent=100 * (b - a) / b))
    write_csv(run_dir / "timing_means.csv", means)
    write_csv(run_dir / "timing_effects.csv", effects)
    lookup = {(r["rank"], r["variant"]): r["mean_us"] for r in means}

    def value(rank, variant):
        result = lookup.get((rank, variant))
        return "pending" if result is None else f"{result:.3f} us"

    text = ["# Hardware-backed timing comparison", "",
            f"{n_trials} separate acquisitions per rank, three circuits per acquisition; rank 1 then rank 58.",
            f"All numbers below are NEW hardware compiler schedule durations, averaged over {n_trials} jobs per rank.",
            "No synthetic values or local scheduling durations are substituted.", "",
            "| Effect | Rank 1 | Rank 58 |", "|---|---|---|",
            "| Classical post-processing versus feed-forward (sequential for both) | "
            + " | ".join(f"{value(r, VARIANTS[1])} versus {value(r, VARIANTS[0])}" for r in (1, 58)) + " |",
            "| Additional parallelisation (sequential to parallel classical) | "
            + " | ".join(f"{value(r, VARIANTS[1])} -> {value(r, VARIANTS[2])}" for r in (1, 58)) + " |", "",
            "## Interpretation", "",
            "The metric is max(T0 + Duration) times the saved hardware backend.dt, with all trace "
            "channels and reported initialization/finalization included. It is not CPU time, queue "
            "time, billed QPU time or a shot-by-shot latency measurement. The older synthetic "
            "preparation-prefix values are a DIFFERENT timing scope and must not be mixed with this table.", "",
            "Within each rank the two sequential variants share their routed native backbone; "
            "the classical variant removes only conditional correction blocks before rescheduling. "
            "The parallel variant starts from the same logical-to-physical layout but may use different "
            "routing and cancellations. Its saving is therefore an end-to-end implementation benefit, "
            "not a pure decomposition into scheduler-only and gate-only contributions.", "",
            "No DD, gate twirling, measurement twirling or M3 calibration jobs are used. "
            "Only the selected analysis basis is acquired; these jobs do not measure entanglement fidelity.", "",
            "Identical timings across trials are expected for frozen circuits and deterministic "
            "compiler schedules. These repetitions do not measure shot-by-shot latency variance. "
            "Negative savings, if observed, are retained rather than concealed.", "",
            f"Timing format and units: {TIMING_GUIDE}", ""]
    (run_dir / "comparison.md").write_text("\n".join(text), encoding="utf-8")
    write_json(run_dir / "manifest.json", manifest)
    print("\n".join(text[:10]), flush=True)


def reference_receiver_correlations(samples, variant, rounds, basis):
    """Apply the accumulated Pauli frame to ONE measured correlation.

    BitArray strings are most-significant-bit first. In each Bell register,
    bit 0 (u) gives the Z byproduct and bit 1 (v) the X byproduct.
    Samples from different registers MUST be aligned shot by shot; their
    separate marginal counts are insufficient for this calculation.
    """
    analysis = [int(bits, 2) for bits in samples["analysis"]]
    totals = [0, 0]
    for channel, name in enumerate("AB"):
        bell = [[int(bits, 2) for bits in samples[f"r{r}_m{name}"]]
                for r in range(1, rounds + 1)]
        if any(len(values) != len(analysis) for values in bell):
            raise ValueError("Unaligned Bell-history samples.")
        for shot, output in enumerate(analysis):
            parity = ((output >> (2 * channel)) ^ (output >> (2 * channel + 1))) & 1
            if variant != VARIANTS[0]:
                for values in bell:
                    u, v = values[shot] & 1, (values[shot] >> 1) & 1
                    parity ^= u if basis == "XX" else (v if basis == "ZZ" else u ^ v)
            totals[channel] += 1 - 2 * parity
    return {"single_basis_correlation_A": totals[0] / len(analysis),
            "single_basis_correlation_B": totals[1] / len(analysis)}


def consume_job(job, entry, manifest, run_dir):
    """Persist the complete result BEFORE trying the experimental parser."""
    result = job.result()
    job_dir = run_dir / f"rank{entry['rank']}_trial{entry['trial']}"
    job_dir.mkdir(exist_ok=True)
    write_json(job_dir / "job_result_metadata.json", result.metadata)
    try:
        write_json(job_dir / "job_metrics.json", job.metrics())
    except Exception as exc:
        # Billing/status metadata is auxiliary, never the timing source.
        (job_dir / "job_metrics_unavailable.txt").write_text(type(exc).__name__, encoding="utf-8")
    if len(result) != len(entry["variant_order"]):
        raise RuntimeError("Unexpected number of PUB results.")
    # Save EVERY PUB first, even if one trace is missing or changes format.
    for variant, pub in zip(entry["variant_order"], result):
        prefix = job_dir / variant
        metadata = pub.metadata or {}
        write_json(prefix.with_suffix(".metadata.json"), metadata)
        counts = {name: data.get_counts() for name, data in pub.data.items()
                  if hasattr(data, "get_counts")}
        write_json(prefix.with_suffix(".counts.json"), counts)
        samples = {name: data.get_bitstrings() for name, data in pub.data.items()
                   if hasattr(data, "get_bitstrings")}
        write_json(prefix.with_suffix(".shots.json"), samples)
    acquired = []
    for index, (variant, pub) in enumerate(zip(entry["variant_order"], result)):
        prefix = job_dir / variant
        metadata = pub.metadata or {}
        timing = metadata.get("compilation", {}).get("scheduler_timing", {}).get("timing")
        if timing is None:
            raise RuntimeError("IBM did not return scheduler_timing. Raw results are saved. "
                               "No further jobs will be submitted and no local fallback is used.")
        raw_text = timing if isinstance(timing, str) else "\n".join(timing)
        prefix.with_suffix(".timing.txt").write_text(raw_text, encoding="utf-8")
        parsed = parse_timing(timing, manifest["backend_snapshot"]["dt_seconds"])
        write_csv(prefix.with_suffix(".instructions.csv"), parsed.pop("instruction_rows"))
        # A conditional schedule without ANY branch/communication evidence is
        # suspicious: do not publish an apparently complete FF duration.
        if variant == VARIANTS[0] and not (parsed["communication_rows"]
                                          or set(parsed["branch_names"]) - {"main"}):
            raise RuntimeError("Feed-forward trace has no branch/communication evidence; "
                               "inspect the saved trace before using its duration.")
        samples = {name: data.get_bitstrings() for name, data in pub.data.items()
                   if hasattr(data, "get_bitstrings")}
        check = reference_receiver_correlations(samples, variant, manifest["settings"]["rounds"],
                                                 manifest["settings"]["analysis_basis"])
        acquired.append(dict(rank=entry["rank"], trial=entry["trial"], variant=variant,
                             pub_index=index, job_id=entry["job_id"], metric=METRIC,
                             dt_seconds=manifest["backend_snapshot"]["dt_seconds"],
                             duration_dt=parsed["duration_dt"], duration_us=parsed["duration_us"],
                             reported_envelope_us=parsed["reported_envelope_us"],
                             communication_rows=parsed["communication_rows"], **check))
    # Commit the matched triple together, avoiding partial or duplicate trials.
    manifest["records"] = [r for r in manifest.get("records", [])
                           if (r["rank"], r["trial"]) != (entry["rank"], entry["trial"])] + acquired
    entry["collected_utc"] = utc_now()
    summarize(run_dir, manifest)


### Hardware acquisition and recovery helpers


In [ ]:
def snapshot_backend(backend):
    target = backend.target
    instructions = []
    for name in sorted(target.operation_names):
        for qubits, props in target[name].items():
            instructions.append(dict(name=name, qubits=list(qubits) if qubits is not None else None,
                                     duration_seconds=getattr(props, "duration", None),
                                     error=getattr(props, "error", None)))
    config = backend.configuration()
    return dict(name=backend.name, captured_utc=utc_now(), num_qubits=backend.num_qubits,
                dt_seconds=float(backend.dt), timing_constraints=getattr(config, "timing_constraints", {}),
                default_rep_delay=getattr(config, "default_rep_delay", None),
                instructions=instructions)


def sampler_configuration(rep_delay=None):
    options = dict(experimental={"execution": {"scheduler_timing": True}},
                   dynamical_decoupling={"enable": False},
                   twirling={"enable_gates": False, "enable_measure": False},
                   execution={"init_qubits": True})
    if rep_delay is not None:
        options["execution"]["rep_delay"] = rep_delay
    return options


def run_hardware(args):
    from qiskit import qpy
    from qiskit_ibm_runtime import QiskitRuntimeService, SamplerV2

    # Connecting is deliberately confined to explicit --run-hardware / --collect.
    service_args = {key: value for key, value in (("name", args.account), ("instance", args.instance))
                    if value is not None}
    service = QiskitRuntimeService(**service_args)
    backend = service.backend(args.backend, use_fractional_gates=False)
    if backend.configuration().simulator:
        raise RuntimeError("This acquisition requires a real backend, not a simulator.")
    if not backend.status().operational:
        raise RuntimeError("Backend is not operational; no jobs submitted.")
    stamp = datetime.now(timezone.utc).strftime("%Y%m%dT%H%M%S_%fZ")
    run_dir = Path(args.output or source_directory() / "timing_runs" / stamp).resolve()
    run_dir.mkdir(parents=True, exist_ok=False)
    manifest = dict(created_utc=utc_now(), versions=dependencies(), sources=list(SOURCE_NAMES),
                    settings=dict(backend=args.backend, rounds=args.rounds, trials=args.trials, shots=args.shots,
                                  optimization_level=args.optimization_level, seed=args.seed,
                                  analysis_basis=args.analysis_basis, dd="none", metric=METRIC),
                    backend_snapshot=snapshot_backend(backend), jobs=[], records=[], circuits={})
    write_json(run_dir / "manifest.json", manifest)
    all_circuits = {}
    layout = args.layout
    # Precompile BOTH ranks before spending hardware credits.
    for rank in (1, 58):
        print(f"Compiling rank {rank}; no hardware jobs submitted yet...", flush=True)
        chosen, circuits = compile_rank(backend, rank, args, layout)
        layout = chosen  # same initial physical assignment for both ranks
        all_circuits[rank] = circuits
        for variant, circuit in circuits.items():
            path = run_dir / f"rank{rank}_{variant}.qpy"
            with path.open("wb") as stream:
                qpy.dump(circuit, stream)
            manifest["circuits"][f"rank{rank}_{variant}"] = dict(
                path=path.name, sha256=hashlib.sha256(path.read_bytes()).hexdigest(),
                initial_layout=chosen, **circuit_stats(circuit))
        write_json(run_dir / "manifest.json", manifest)

    # Same repetition delay for all variants; not interpreted as circuit time.
    options = sampler_configuration(args.rep_delay)
    sampler = SamplerV2(mode=backend, options=options)
    write_json(run_dir / "sampler_options.json", options)
    print(f"Submitting {2 * args.trials} jobs, 3 circuits/job, {args.shots} shots/circuit. "
          f"Outputs: {run_dir}", flush=True)
    for rank in (1, 58):
        for trial in range(1, args.trials + 1):
            order = list(VARIANTS if trial % 2 else reversed(VARIANTS))
            print(f"Rank {rank}, trial {trial}/{args.trials}: {', '.join(order)}", flush=True)
            # Do NOT retry submissions automatically: a network timeout can
            # leave a paid job running. IDs are saved before awaiting results.
            job = sampler.run([all_circuits[rank][variant] for variant in order], shots=args.shots)
            entry = dict(rank=rank, trial=trial, variant_order=order,
                         job_id=job.job_id(), submitted_utc=utc_now())
            manifest["jobs"].append(entry)
            write_json(run_dir / "manifest.json", manifest)
            print(f"Saved job {job.job_id()}; waiting for result...", flush=True)
            try:
                consume_job(job, entry, manifest, run_dir)
            except Exception:
                print(f"Stopped without submitting further jobs. Saved run directory: {run_dir}",
                      file=sys.stderr)
                if globals().get("__file__"):
                    print(f'python "{Path(__file__).resolve()}" --collect "{run_dir}"', file=sys.stderr)
                else:
                    print("Use the notebook recovery cell with this directory. "
                          "Recovery does not submit new jobs.", file=sys.stderr)
                raise
    print(f"\nComplete: {run_dir / 'comparison.md'}", flush=True)
    return run_dir


def collect_saved(args):
    from qiskit_ibm_runtime import QiskitRuntimeService

    run_dir = Path(args.collect).resolve()
    manifest = json.loads((run_dir / "manifest.json").read_text(encoding="utf-8"))
    kwargs = {key: value for key, value in (("name", args.account), ("instance", args.instance))
              if value is not None}
    service = QiskitRuntimeService(**kwargs)
    for entry in manifest["jobs"]:
        consume_job(service.job(entry["job_id"]), entry, manifest, run_dir)
    print("Retrieved recorded jobs only. Missing acquisitions were NOT resubmitted.")
    return run_dir


### Optional offline validation helper


In [ ]:
def self_test():
    """Purely local tests: parser, ideal teleportation, sparse native compilation."""
    dependencies()
    from qiskit import transpile, qpy
    from qiskit_aer import AerSimulator
    from qiskit.providers.fake_provider import GenericBackendV2
    from qiskit.transpiler import CouplingMap, InstructionProperties
    from qiskit_ibm_runtime.circuit import MidCircuitMeasure
    from qiskit_ibm_runtime.options import SamplerOptions
    from qiskit.circuit import Measure

    # Validate real SDK options without constructing a service or Sampler.
    checked_options = SamplerOptions(**sampler_configuration())
    assert checked_options.experimental["execution"]["scheduler_timing"] is True

    fixture = ("main,x_0,Qubit 0,10,5,play\n"
               "main,broadcast,Hub,15,50,broadcast\n"
               "then,x_1,Qubit 1,65,5,play\n"
               "else,delay,Qubit 1,65,5,delay\n"
               "main,FINI_0,AWGR0_0,70,20,capture\n")
    parsed = parse_timing(fixture, 4e-9)
    assert parsed["duration_dt"] == 90
    assert math.isclose(parsed["duration_us"], 0.36)
    assert parsed == parse_timing(fixture.splitlines(), 4e-9)
    for bad in ("", "bad,record", "main,x,Qubit 0,-1,2,play"):
        try:
            parse_timing(bad, 4e-9)
        except ValueError:
            pass
        else:
            raise AssertionError("Malformed trace was accepted.")

    simulator = AerSimulator(method="stabilizer", max_parallel_threads=2)

    def check_ideal(circuit, variant, rounds, basis, engine=None):
        engine = simulator if engine is None else engine
        counts = engine.run(transpile(circuit, engine, optimization_level=0),
                            shots=64, seed_simulator=271).result().get_counts()
        for sample in counts:
            bits = [int(bit) for bit in sample.replace(" ", "")[::-1]]
            registers = {register.name: ["".join(str(bits[circuit.find_bit(bit).index])
                                                 for bit in reversed(register))]
                         for register in circuit.cregs}
            correlations = reference_receiver_correlations(registers, variant, rounds, basis)
            assert all(value == (-1 if basis == "YY" else 1)
                       for value in correlations.values()), (circuit.name, basis, correlations)

    ideal_count = 0
    for rank in (1, 58):
        for rounds in (1, 3):
            for variant in VARIANTS:
                for basis in ("XX", "YY", "ZZ"):
                    circuit = build_circuit(rank, variant, rounds, basis)
                    assert circuit.num_qubits == 4 + 5 * rounds
                    assert circuit.count_ops()["measure"] == 4 * rounds + 4
                    assert circuit.count_ops().get("if_else", 0) == (
                        4 * rounds if variant == VARIANTS[0] else 0)
                    check_ideal(circuit, variant, rounds, basis)
                    ideal_count += 1
    print(f"PASS: {ideal_count} ideal circuits; both channels recover all signed Bell correlations.")

    # This backend is ONLY a compilation test; its random timings NEVER
    # enter a report or the hardware comparison.
    backend = GenericBackendV2(19, basis_gates=["id", "rz", "sx", "x", "cz"],
                               coupling_map=CouplingMap.from_line(19), control_flow=True, seed=43)
    backend.target.add_instruction(MidCircuitMeasure(), {
        (q,): InstructionProperties(duration=6400 * backend.dt, error=0.01) for q in range(19)})
    # Native two-qubit block resynthesis can introduce individually non-Clifford
    # rotations even for an overall Clifford circuit. Use an exact, untruncated
    # MPS simulation for these native circuits, not the stabilizer-only engine.
    native_simulator = AerSimulator(method="matrix_product_state", max_parallel_threads=2,
                                    matrix_product_state_truncation_threshold=0)
    test_args = argparse.Namespace(rounds=3, analysis_basis="ZZ", optimization_level=2, seed=42)
    layout = None
    for rank in (1, 58):
        layout, circuits = compile_rank(backend, rank, test_args, layout)
        for variant, circuit in circuits.items():
            buffer = io.BytesIO()
            qpy.dump(circuit, buffer)
            buffer.seek(0)
            restored = qpy.load(buffer)[0]
            assert restored.count_ops() == circuit.count_ops()
            assert len(circuit_stats(circuit)["used_physical_qubits"]) <= 19
            ideal_proxy = restored.copy()
            ideal_proxy.data = [item.replace(operation=Measure())
                                if item.operation.name == "measure_2" else item
                                for item in restored.data if item.operation.name != "delay"]
            check_ideal(ideal_proxy, variant, 3, "ZZ", engine=native_simulator)
            print(f"PASS: sparse native compile / IBM scheduling / QPY / ideal ZZ: rank {rank}, {variant}")
    print("PASS: strict trace parser and dt conversion. No service connection or hardware submission.")


## 3. Check settings and installed packages

Requires `qiskit >= 2.1`, `qiskit-ibm-runtime >= 0.43.0` and a normal IPython
kernel. The optional offline tests also require `qiskit-aer`. This cell only
checks the local environment and configuration; it does not connect to IBM.


In [ ]:
assert isinstance(N_TRIALS, int) and not isinstance(N_TRIALS, bool) and N_TRIALS > 0
assert isinstance(SHOTS, int) and not isinstance(SHOTS, bool) and SHOTS > 0
assert ROUNDS in (1, 2, 3)
assert OPTIMIZATION_LEVEL in (2, 3)
assert ANALYSIS_BASIS in ("XX", "YY", "ZZ")
assert not (RUN_HARDWARE and RUN_RECOVERY), "Choose a new acquisition OR recovery, not both."
if INITIAL_LAYOUT is not None:
    assert len(INITIAL_LAYOUT) == 4 + 5 * ROUNDS
    assert len(set(INITIAL_LAYOUT)) == len(INITIAL_LAYOUT)
    assert all(isinstance(q, int) and q >= 0 for q in INITIAL_LAYOUT)
if REP_DELAY_SECONDS is not None:
    assert REP_DELAY_SECONDS > 0

VERSIONS = dependencies()
BENCHMARK_ARGS = argparse.Namespace(
    backend=BACKEND_NAME, trials=N_TRIALS, rounds=ROUNDS, shots=SHOTS,
    optimization_level=OPTIMIZATION_LEVEL, analysis_basis=ANALYSIS_BASIS,
    seed=SEED_TRANSPILER, account=ACCOUNT_NAME, instance=RUNTIME_INSTANCE,
    layout=INITIAL_LAYOUT, rep_delay=REP_DELAY_SECONDS,
    output=str(OUTPUT_DIRECTORY) if OUTPUT_DIRECTORY is not None else None,
)
print("Installed versions:", VERSIONS)
print(f"Plan: rank 1 then rank 58; {N_TRIALS} trials/rank, "
      f"{2*N_TRIALS} jobs, {6*N_TRIALS} circuit executions, {SHOTS} shots/circuit.")
print(f"R{ROUNDS}; optimisation level {OPTIMIZATION_LEVEL}; no DD; basis {ANALYSIS_BASIS}.")
print("Hardware submission enabled:", RUN_HARDWARE)


## 4. Optional offline checks � no hardware jobs


In [ ]:
if RUN_OFFLINE_TESTS:
    self_test()
else:
    print("Offline physics/compilation tests skipped. Set RUN_OFFLINE_TESTS = True to run them.")


## 5. Acquire hardware timings

This cell remains inactive while `RUN_HARDWARE` is `False`. When enabled, it
compiles both ranks before submitting jobs, then completes rank 1's eight
acquisitions before starting rank 58. Each job ID is saved before waiting for
its result. There are no automatic submission retries.

If timing metadata is absent or unsupported, the run stops rather than
inventing a local duration. Re-fetch saved jobs with the recovery cell below.
The run directory is shown before submission. A kernel-level guard prevents
accidentally submitting the entire acquisition twice by rerunning this cell.
Restart the kernel for an intentionally new acquisition.


In [ ]:
if RUN_HARDWARE:
    if globals().get("_HARDWARE_RUN_ATTEMPTED", False):
        raise RuntimeError("This kernel already attempted an acquisition. "
                           "Use recovery for existing jobs, or restart the kernel for a new run.")
    # Choose and retain the destination BEFORE connecting or submitting.
    stamp = datetime.now(timezone.utc).strftime("%Y%m%dT%H%M%S_%fZ")
    LAST_RUN_DIRECTORY = Path(OUTPUT_DIRECTORY or Path.cwd() / "timing_runs" / stamp).resolve()
    BENCHMARK_ARGS.output = str(LAST_RUN_DIRECTORY)
    print("Run directory:", LAST_RUN_DIRECTORY)
    _HARDWARE_RUN_ATTEMPTED = True
    run_hardware(BENCHMARK_ARGS)
else:
    print("No IBM connection or hardware submission. Set RUN_HARDWARE = True intentionally to acquire data.")


## 6. Optional recovery � no new submissions

Set `RUN_HARDWARE = False`, `RUN_RECOVERY = True` and
`RECOVER_RUN_DIRECTORY` to the saved run directory, then rerun configuration,
preflight and this cell. Recovery only retrieves recorded job IDs. It does not
submit missing acquisitions. Final averages remain pending until all requested
trials for a rank/method are available. Older two-trial runs retain their
original trial count rather than being relabelled as eight-trial results.


In [ ]:
if RUN_RECOVERY:
    if RECOVER_RUN_DIRECTORY is None:
        raise ValueError("Set RECOVER_RUN_DIRECTORY to an existing run directory.")
    recovery_args = argparse.Namespace(
        collect=str(RECOVER_RUN_DIRECTORY), account=ACCOUNT_NAME, instance=RUNTIME_INSTANCE)
    LAST_RUN_DIRECTORY = collect_saved(recovery_args)
else:
    print("Recovery disabled.")


## 7. Display the comparison and averages

The run directory contains `comparison.md`, `timings_per_trial.csv`,
`timing_means.csv`, `timing_effects.csv` and `manifest.json`, plus the QPY
circuits and raw metadata, timing traces and shot histories for each job.
Savings are calculated within matched trials and then averaged; negative
savings are retained. Rank/method means are reported only when all requested
trials are present. No confidence interval for physical latency is inferred
from deterministic compiler schedule repetitions.


In [ ]:
if LAST_RUN_DIRECTORY is None:
    display(Markdown("No acquisition has been run. Hardware submission is disabled by default."))
else:
    result_directory = Path(LAST_RUN_DIRECTORY)
    report = result_directory / "comparison.md"
    print("Results:", result_directory)
    if report.exists():
        display(Markdown(report.read_text(encoding="utf-8")))
    else:
        print("No completed timing triples yet. Check the saved manifest / job IDs.")
    for filename in ("timing_means.csv", "timing_effects.csv"):
        path = result_directory / filename
        if path.exists():
            with path.open(newline="", encoding="utf-8") as stream:
                rows = list(csv.reader(stream))
            if rows:
                display(Markdown("### " + filename))
                header, *body = rows
                table = ["| " + " | ".join(header) + " |",
                         "| " + " | ".join("---" for _ in header) + " |"]
                table += ["| " + " | ".join(row) + " |" for row in body]
                display(Markdown("\n".join(table)))
